In [ ]:
import streamlit as st
import bot
import tools
from dotenv import load_dotenv



In [ ]:
load_dotenv()

In [ ]:
st.set_page_config(page_title="Institutional AI Terminal", layout="wide")

In [ ]:
# --- SIDEBAR: LIVE DATA ---

In [ ]:
st.sidebar.title("🏛️ Portfolio Monitor")

In [ ]:
try:
    acc = tools.get_account_status()
    st.sidebar.metric("Paper Cash", f"${acc['cash']:,.2f}")
    st.sidebar.metric("Buying Power", f"${acc['buying_power']:,.2f}")
except:
    st.sidebar.error("API Connection Offline")


In [ ]:
# --- MAIN UI ---
st.title("Agentic Multi-Asset Terminal")
st.info("Currently operating in **Paper Trading Mode** (Zero Financial Risk)")

col1, col2 = st.columns([1, 1])

with col1:
    st.subheader("1. AI Market Analysis")
    asset = st.text_input("Ticker (e.g. BTC/USD, TSLA, ETH/USD)", "BTC/USD")
    risk_val = st.slider("Institutional Risk Tolerance", 0, 100, 20)
    context = st.text_area("Market Intelligence", "Analyzing 4h charts and recent FOMC sentiment.")

    if st.button("🧠 Run AI Agent"):
        with st.spinner("Agent generating risk-adjusted proposal..."):
            res = bot.get_institutional_analysis(asset, context, risk_val)
            st.session_state.current_analysis = res

with col2:
    st.subheader("2. Human-in-the-Loop Execution")
    if "current_analysis" in st.session_state:
        res = st.session_state.current_analysis
        
        # Professional UI for the Proposal
        with st.container(border=True):
            st.write(f"**Recommendation:** :{ 'green' if res['action'] == 'BUY' else 'red' if res['action'] == 'SELL' else 'orange' }[{res['action']}]")
            st.markdown(f"**Risk Factor:** {res['risk_assessment']}")
            st.caption(f"Reasoning: {res['institutional_reasoning']}")
            
            if res['action'] != "HOLD":
                # APPROVAL BUTTON
                if st.button(f"Confirm & Execute {res['action']} Order"):
                    try:
                        # 1. Physical Execution
                        tools.execute_bracket_order(asset, res['qty'], res['action'], res['tp'], res['sl'], "/" in asset)
                        # 2. Log for History
                        tools.log_trade(asset, res['action'], res['qty'], res['institutional_reasoning'])
                        st.success("Order filled with Bracket Protection.")
                        st.balloons()
                    except Exception as e:
                        st.error(f"Execution Failed: {e}")
            else:
                st.warning("Agent advises 'No Trade' under current risk parameters.")




In [ ]:
# --- SECTION 3: TRADE HISTORY ---
st.markdown("---")
st.subheader("📜 Audit Trail (Trade History)")
history = tools.get_logs()
if not history.empty:
    st.dataframe(history, use_container_width=True)
else:
    st.write("No trades recorded yet. Run an analysis and approve a trade to see the log.")